# Assessment: judge the evidence against the test steps

**What it does**
Runs the assessment for one case and shows the result:
1. Takes the case's newest interpretation, which holds its test steps.
2. Converts and splits the case's evidence into parts (notebook 10).
3. **Loop**: one AI call per part. Each call gets the control details and test plan PDFs, all test steps and that part. The answer per test step is supports / contradicts / partly / not relevant, plus each criterion as met / not met / not shown, the failing items, and where in the evidence it saw them.
4. **Reference check**: every reference is checked against the evidence sent (does the document, page or row exist). Problems are kept and shown.
5. **Combine**: one AI call turns all findings into **effective / not effective** per test step, with reason, exceptions and references. "No evidence" is flagged.
6. **Save and report**: the verdicts are saved (and each test step's status), and a **read-only PDF** is written to `storage/outputs/<case>/` with its SHA-256 recorded.

**Why**
This is the goal of KCT: a traceable verdict per test step, backed by the evidence. The logic is in `AssessmentService` (`services/assessment/`). The API runs the same code in the background: `POST /api/v1/cases/{case}/assessments`.

**How to use**
1. The case needs an **interpretation** (`04_demo_data.ipynb`) and **evidence** (`10_evidence_case.ipynb`, or Step 1 here).
2. Set `KCT_DB_DRIVER=sqlite` in `.env` to keep the results after the kernel stops.
3. Run all cells. With `KCT_AI_PROVIDER=fake` it costs nothing; the answers then come from `tests/fixtures/fake_ai/`, so they are the same for any evidence. With `gemini`, Step 2 shows the size first: check it before Step 3.

## Setup

Picks the newest case that has an interpretation; set `CASE` to choose another.

In [ ]:
import os, sys
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from services.app import boot
from services.config import settings

kct = boot()
config = settings()
with_steps = sorted({m.case_reference for m in kct.database.interpretations()})
CASE = with_steps[-1] if with_steps else None     # or e.g. "case-002"

print("AI provider:", config.ai_provider, "/", config.ai_model, "(fake = no cost)" if config.ai_provider == "fake" else "(real calls, costs money)")
print("Database:   ", config.db_driver)
print("Case:       ", CASE or "❌ none with an interpretation: run 04_demo_data.ipynb first")

## Step 1: Evidence for the case

Lists the case's evidence. If there is none, `ADD_SAMPLES = True` puts fake evidence of every kind into the case: Word, Excel, an email thread and a 70-page PDF.

In [ ]:
from factories import DocxFactory, ExcelFactory, MsgFactory, PdfFactory
from services.files.file_service import UploadedFile

ADD_SAMPLES = True

evidence = kct.evidences.case(CASE)
if not evidence.evidence_files() and ADD_SAMPLES:
    for factory in (DocxFactory(), ExcelFactory(rows=60), MsgFactory(history=3, images=2), PdfFactory(pages=70)):
        evidence.store([UploadedFile(name=factory.name, data=factory.make())])
    print("Added fake evidence")
for relative in evidence.evidence_files():
    print(f"   {relative}")
print(f"{len(evidence.evidence_files())} evidence file(s)" if evidence.evidence_files() else "No evidence: every step will be not effective (no evidence)")

## Step 2: Size before sending

How many AI calls the run makes and roughly how many tokens each takes. The control details and test plan PDFs are sent with every call, so their pages count in each one.

In [ ]:
from services.control_testing.data import DocumentKind
from services.evidences.evidence_splitter import EvidenceSplitter

context = kct.files.documents(CASE, DocumentKind.CONTROL_DETAILS) + kct.files.documents(CASE, DocumentKind.TEST_PLAN)
context_tokens = sum(EvidenceSplitter.tokens(d) for d in context)
parts = evidence.parts()
for part in parts:
    print(f"   {part.label:<70} ~{part.estimated_tokens + context_tokens:>9,} tokens")
total = sum(p.estimated_tokens for p in parts) + context_tokens * len(parts)
print(f"\n{len(parts)} evidence call(s) + 1 verdict call, ~{total:,} input tokens in total "
      f"(control details + test plan: ~{context_tokens:,} per call)")

## Step 3: Run the assessment

Runs here and prints progress per part. It takes a few seconds with `fake`, minutes with Gemini. A part that fails is recorded and the run goes on.

In [ ]:
result = kct.assessments.assess(CASE, progress=print)
meta = result.metadata
print("\nAssessment:", meta.assessment_id, "-", meta.status)

## Step 4: Verdicts

One row per test step. `no evidence` = no evidence said anything about the step. `fallback` = the model gave no verdict, so the application set not effective.

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.DataFrame([
    {"step": v.step_reference, "title": v.title, "verdict": v.verdict.replace("_", " "),
     "no evidence": v.no_evidence, "fallback": v.fallback, "exceptions": len(v.exceptions), "reason": v.reason}
    for v in result.verdicts
])

## Step 5: Exceptions and reference problems

The failing items per test step, and references the model gave that did not match the evidence sent.

In [ ]:
for v in result.verdicts:
    for e in v.exceptions:
        where = f"{e.reference.document}, {e.reference.location}" if e.reference else "-"
        print(f"{v.step_reference}  {e.item}: {e.description}\n      at {where}")
problems = [(p.label, f.step_reference, x) for p in result.parts for f in p.findings for x in f.reference_problems]
print(f"\n{len(problems)} reference problem(s) in the findings")
for label, step, text in problems[:10]:
    print(f"   {label} / {step}: {text}")

## Step 6: The report

The read-only PDF and its SHA-256. Open it from `storage/outputs/<case>/` (VS Code: right-click > Reveal in Finder). The API serves it at `GET /api/v1/assessments/{id}/report.pdf`.

In [ ]:
import hashlib

if meta.report_path:
    report = kct.assessments.report(meta.assessment_id)
    print("Report: ", config.storage_root / meta.report_path if config.files_driver == "local" else meta.report_path)
    print("SHA-256:", meta.report_sha256)
    print("Matches:", hashlib.sha256(report).hexdigest() == meta.report_sha256)

## Step 7: Earlier runs of this case

Every run is kept; the test steps' status follows the newest one.

In [ ]:
for m in kct.assessments.index(CASE):
    print(f"{m.created_at[:19]}  {m.assessment_id}  {m.status:<9}  parts {m.parts_done}/{m.parts_total}"
          + (f", {m.parts_failed} failed" if m.parts_failed else ""))

## The same through the API

```bash
curl -X POST http://localhost:8000/api/v1/cases/case-002/assessments          # returns at once: status queued
curl http://localhost:8000/api/v1/assessments/<assessment_id>                  # progress, then verdicts
curl -o report.pdf http://localhost:8000/api/v1/assessments/<assessment_id>/report.pdf
```